# ASSIGNMENT 5 **Word Embeddings, Order, and the Road to RNNs**

## Part B: (Code) Tokenization and Vocabulary Exploration
В этой части я делаю токенизацию и анализ словаря, чтобы увидеть, как текст превращается в данные. Это помогает понять, как предобработка влияет на модель.

B1. Compare tokenization

In [5]:
import spacy

# загружаем модель spaCy (она умеет умно разбивать текст)
nlp = spacy.load("en_core_web_sm")

# наши предложения из разных категорий
sentences = [
    "The film was surprisingly boring.",
    "I never liked this movie.",
    "I didn't say it was bad.",
    "Why is this app not working?",
    "Turn off the lights immediately.",
    "Microsoft announced a partnership with OpenAI in 2024.",
    "Flights from Astana to Dubai are expensive.",
    "nah bro this ain't it 😭"
]

for text in sentences:
    print("\n====================")

    # 1. Raw text — исходное предложение
    print("Raw text:", text)

    # 2. Simple split — просто делим по пробелам (очень простой метод) проблема: не убирает пунктуацию и не понимает язык
    split_tokens = text.split()
    print("Split tokens:", split_tokens)


    # 3. spaCy tokenization — умный способ
    doc = nlp(text)
    spacy_tokens = [token.text for token in doc]
    print("spaCy tokens:", spacy_tokens)
    # отделяет пунктуацию
    # разбивает "didn't" → "did" + "n't"

    # 4. Lowercase — приводим всё к маленьким буквам
    lower_tokens = [token.text.lower() for token in doc]
    print("Lowercase:", lower_tokens)
    # помогает считать слова одинаковыми (Movie = movie)

    # 5. Filtering — убираем мусор
    filtered_tokens = [
        token.text.lower()
        for token in doc
        if not token.is_stop and not token.is_punct
    ]
    print("Filtered:", filtered_tokens)
    # убираем stopwords (the, is, was)
    # убираем пунктуацию
    # остаются только важные слова

    # 6. Lemmatization — приводим к базовой форме
    lemmas = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop and not token.is_punct
    ]
    print("Lemmas:", lemmas)
    # liked → like
    # running → run


Raw text: The film was surprisingly boring.
Split tokens: ['The', 'film', 'was', 'surprisingly', 'boring.']
spaCy tokens: ['The', 'film', 'was', 'surprisingly', 'boring', '.']
Lowercase: ['the', 'film', 'was', 'surprisingly', 'boring', '.']
Filtered: ['film', 'surprisingly', 'boring']
Lemmas: ['film', 'surprisingly', 'boring']

Raw text: I never liked this movie.
Split tokens: ['I', 'never', 'liked', 'this', 'movie.']
spaCy tokens: ['I', 'never', 'liked', 'this', 'movie', '.']
Lowercase: ['i', 'never', 'liked', 'this', 'movie', '.']
Filtered: ['liked', 'movie']
Lemmas: ['like', 'movie']

Raw text: I didn't say it was bad.
Split tokens: ['I', "didn't", 'say', 'it', 'was', 'bad.']
spaCy tokens: ['I', 'did', "n't", 'say', 'it', 'was', 'bad', '.']
Lowercase: ['i', 'did', "n't", 'say', 'it', 'was', 'bad', '.']
Filtered: ['bad']
Lemmas: ['bad']

Raw text: Why is this app not working?
Split tokens: ['Why', 'is', 'this', 'app', 'not', 'working?']
spaCy tokens: ['Why', 'is', 'this', 'app', 'no

Токенизация — это процесс разбиения текста на более мелкие единицы, называемые токенами. Простой способ — разделение по пробелам, но он недостаточен, так как не учитывает пунктуацию и сокращения. Библиотека spaCy выполняет более продвинутую токенизацию: она отделяет знаки препинания и учитывает структуру языка. Приведение к нижнему регистру помогает нормализовать текст. Фильтрация удаляет незначимые слова, а лемматизация приводит слова к их базовой форме.

## B2. Vocabulary observations

In [6]:
from collections import Counter

# создаём список для всех токенов
all_tokens = []

# снова проходимся по тем же предложениям
for text in sentences:
    doc = nlp(text)

    # берём очищенные токены:
    # - lowercase
    # - без stopwords
    # - без пунктуации
    tokens = [
        token.text.lower()
        for token in doc
        if not token.is_stop and not token.is_punct
    ]

    all_tokens.extend(tokens)

# Статистика
# общее количество токенов
total_tokens = len(all_tokens)
# количество уникальных токенов
unique_tokens = len(set(all_tokens))

print("Vocabulary Statistics")
print("Total tokens:", total_tokens)
print("Unique tokens:", unique_tokens)

# Топ-5 самых частых слов
counter = Counter(all_tokens)
print("Top 5 most frequent tokens:")
for word, freq in counter.most_common(5):
    print(word, ":", freq)

# Примеры "мусорных" слов
print("\nNoisy tokens examples:")
print(["nah", "bro", "😭", "this", "was"])

# Примеры, где важна токенизация
print("\nTokenization matters examples:")

examples = ["didn't", "OpenAI", "?"]

for ex in examples:
    doc = nlp(ex)
    print(ex, "->", [token.text for token in doc])

Vocabulary Statistics
Total tokens: 24
Unique tokens: 24
Top 5 most frequent tokens:
film : 1
surprisingly : 1
boring : 1
liked : 1
movie : 1

Noisy tokens examples:
['nah', 'bro', '😭', 'this', 'was']

Tokenization matters examples:
didn't -> ['did', "n't"]
OpenAI -> ['OpenAI']
? -> ['?']


Набор данных содержит небольшое количество токенов, так как используется ограниченное число предложений. Некоторые токены, например "nah", "bro" и эмодзи, являются шумом и не несут полезной информации. Токенизация особенно важна в случаях, таких как "didn't", которое разбивается на "did" и "n't", именованные сущности, например "OpenAI", а также знаки препинания, такие как "?".

## B3. Word-vector exploration

In [9]:
# загружаем pretrained модель spaCy
# ВАЖНО: именно "md", потому что в ней есть вектора слов
!python -m spacy download en_core_web_md
nlp = spacy.load("en_core_web_md")

# выбираем слова из разных категорий (эмоции, фильмы, путешествия)
words = [
    "happy", "sad", "angry",
    "actor", "film", "director",
    "plane", "train", "airport"
]

# список слов для сравнения (чтобы избежать шума из всего словаря)
candidates = [
    "joyful", "glad", "unhappy", "depressed", "furious",
    "movie", "cinema", "actor", "director",
    "travel", "trip", "aircraft", "flight", "airport"
]

print("B3 WORD VECTORS")

# проходим по каждому слову
for w in words:
    token = nlp.vocab[w]

    print("\n--------------------")
    print("Word:", w)

    # проверяем, есть ли у слова вектор
    print("Has vector:", token.has_vector)

    # если вектор есть — ищем похожие слова
    if token.has_vector:
        sims = []

        # сравниваем текущее слово с каждым словом из candidates
        for c in candidates:
            # считаем similarity (насколько слова похожи)
            similarity_score = token.similarity(nlp.vocab[c])

            # сохраняем результат (слово + степень похожести)
            sims.append((c, similarity_score))

        # сортируем по убыванию похожести
        sims = sorted(sims, key=lambda x: -x[1])

        print("Nearest neighbors:")

        # выводим 5 самых похожих слов
        for c, s in sims[:5]:
            print(c, "->", round(s, 3))

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_md-3.8.0/en_core_web_md-3.8.0-py3-none-any.whl (33.5 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.

========== B3 WORD VECTORS ==========

--------------------
Word: happy
Has vector: True
Nearest neighbors:
joyful -> 0.734
unhappy -> 0.469
depressed -> 0.469
furious -> 0.469
glad -> 0.446

--------------------
Word: sad
Has vector: True
Nearest neighbors:
glad -> 0.594
unhappy -> 0.535
depressed -> 0.535
furious -> 0.535
joyful -> 0.383

--------------------
Word: angry
Has vector: True
Nearest neighbors:
unhappy -> 1.0
depressed -> 1.0
furious -> 1.0
glad -> 0.733
joyful -> 0.376

--------------------
Word: ac

| Word     | Has vector | Nearest neighbors |
|----------|------------|------------------|
| happy    | Yes        | joyful, unhappy, depressed |
| sad      | Yes        | glad, unhappy, depressed |
| angry    | Yes        | unhappy, depressed, furious |
| actor    | Yes        | movie, unhappy, depressed |
| film     | Yes        | movie, glad, unhappy |
| director | Yes        | travel, cinema, unhappy |
| plane    | Yes        | aircraft, cinema, movie |
| train    | Yes        | travel, cinema, unhappy |
| airport  | Yes        | flight, cinema, travel |

Большинство слов имеют предобученные вектора, что подтверждает, что модель поддерживает семантические представления. Некоторые ближайшие слова являются осмысленными, например, "happy" близко к "joyful", "film" к "movie", а "plane" к "aircraft". Это показывает, что эмбеддинги способны отражать семантическую схожесть между словами.

Однако не все результаты являются точными. Некоторые соседние слова не связаны по смыслу, например, "actor" оказывается близким к "unhappy", а "train" к "depressed". Это происходит потому, что сравнение выполняется в рамках ограниченного набора слов (candidates), который может не содержать действительно подходящих вариантов. В результате модель выбирает наиболее близкое доступное слово, даже если оно не является семантически корректным.

Это показывает, что эмбеддинги полезны, но их качество зависит от выбранного пространства сравнения, и без контекста они не могут полностью передать смысл.

# Part C: (Code) Static Embedding Baseline for Sentiment Classification
В этой части я строю модель классификации тональности с помощью усреднённых эмбеддингов, чтобы увидеть, как они работают на практике. Это показывает, как текст превращается в вектора и используется в модели.

## C1. Load and inspect the dataset

In [10]:
!pip install datasets

In [11]:
from datasets import load_dataset
import numpy as np

# загружаем датасет IMDB (отзывы на фильмы)
dataset = load_dataset("imdb")

# делим на train и test
train_data = dataset["train"]
test_data = dataset["test"]

# выводим размер данных
print("Train size:", len(train_data))
print("Test size:", len(test_data))

# Показываем примеры
print("\n--- Positive examples ---")

pos_count = 0
for example in train_data:
    if example["label"] == 1:  # 1 = positive
        print(example["text"][:200])  # первые 200 символов
        pos_count += 1
        if pos_count == 2:
            break


print("\n--- Negative examples ---")

neg_count = 0
for example in train_data:
    if example["label"] == 0:  # 0 = negative
        print(example["text"][:200])
        neg_count += 1
        if neg_count == 2:
            break

# Средняя длина отзывов
lengths = []

for example in train_data:
    # делим текст на слова (простая токенизация)
    tokens = example["text"].split()
    lengths.append(len(tokens))

# считаем среднее
avg_length = np.mean(lengths)

print("\nAverage review length (in tokens):", avg_length)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Train size: 25000
Test size: 25000

--- Positive examples ---
Zentropa has much in common with The Third Man, another noir-like film set among the rubble of postwar Europe. Like TTM, there is much inventive camera work. There is an innocent American who gets emo
Zentropa is the most original movie I've seen in years. If you like unique thrillers that are influenced by film noir, then this is just the right cure for all of those Hollywood summer blockbusters c

--- Negative examples ---
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev
"I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on any level. As for the claim that 

Average review length (in tokens): 233.7872


| Действие | Наблюдение |
|---------|-----------|
| Загрузка IMDB dataset | Данные сбалансированы (25k / 25k) |
| Анализ размеров | Достаточно данных для обучения модели |
| Просмотр примеров | Чётко виден sentiment в тексте |
| Средняя длина | Длинные тексты → сложнее обработка |

## C2 — Preprocessing

In [12]:
import spacy

# загружаем модель spaCy
nlp = spacy.load("en_core_web_sm")

def preprocess(text):
    # обрабатываем текст
    doc = nlp(text)

    # токенизация + очистка
    tokens = [
        token.text.lower()      # lowercase
        for token in doc
        if not token.is_punct   # убираем пунктуацию
        # stopwords оставляем (важно для negation)
        # лемматизацию не используем
    ]

    return tokens


# новый пример (НЕ из задания)
example = "This movie was not interesting at all, I expected more."
print("Original:", example)
print("Processed:", preprocess(example))

Original: This movie was not interesting at all, I expected more.
Processed: ['this', 'movie', 'was', 'not', 'interesting', 'at', 'all', 'i', 'expected', 'more']


Мы используем spaCy для токенизации текста. Все слова приводим к нижнему регистру для единообразия. Удаляем пунктуацию, так как она не влияет на определение тональности. Стоп-слова не удаляем, поскольку они важны для передачи отрицания (например, "not"). Лемматизацию не применяем, чтобы сохранить исходные формы слов.

## C3. Create one vector per review

In [13]:
import spacy
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

# загружаем модель с предобученными эмбеддингами
# (она уже знает значения слов → vectors)
nlp = spacy.load("en_core_web_md")

# 1. MEAN POOLING (baseline)

def mean_pooling(text):
    """
    Превращаем текст в один вектор:
    берём все вектора слов и считаем среднее
    """
    doc = nlp(text)  # spaCy разбивает текст на токены

    vectors = []

    for token in doc:
        # берём только слова с векторами
        # и убираем пунктуацию
        if token.has_vector and not token.is_punct:
            vectors.append(token.vector)

    # если есть слова → считаем среднее
    if len(vectors) > 0:
        return np.mean(vectors, axis=0)

    # если вдруг нет слов → возвращаем нули
    return np.zeros(nlp.vocab.vectors_length)

# 2. MAX POOLING
def max_pooling(text):
    """
    Берём максимальные значения по каждому измерению
    → выделяет "сильные" слова
    """
    doc = nlp(text)

    vectors = [
        token.vector
        for token in doc
        if token.has_vector and not token.is_punct
    ]

    if len(vectors) > 0:
        return np.max(vectors, axis=0)

    return np.zeros(nlp.vocab.vectors_length)

# 3. MEAN + MAX (комбинированный метод)
def mean_max_pooling(text):
    """
    Объединяем:
    - среднее значение (общий смысл)
    - максимум (важные слова)
    """
    doc = nlp(text)

    vectors = [
        token.vector
        for token in doc
        if token.has_vector and not token.is_punct
    ]

    if len(vectors) > 0:
        mean_vec = np.mean(vectors, axis=0)
        max_vec = np.max(vectors, axis=0)

        # объединяем два вектора в один длинный
        return np.concatenate([mean_vec, max_vec])

    # если нет слов → нули (размер ×2)
    size = nlp.vocab.vectors_length
    return np.zeros(size * 2)

# 4. TF-IDF WEIGHTED AVERAGE
# сначала обучаем TF-IDF на части данных (он определяет важность слов)
texts_for_tfidf = [x["text"] for x in train_data.select(range(2000))]

tfidf = TfidfVectorizer(max_features=5000)
tfidf.fit(texts_for_tfidf)

# создаём словарь: слово → важность
idf_dict = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))


def tfidf_weighted(text):
    """
    Учитываем важность слов:
    важные слова → больший вес
    """
    doc = nlp(text)

    vectors = []
    weights = []

    for token in doc:
        if token.has_vector and not token.is_punct:
            word = token.text.lower()

            # получаем вес слова (если нет — 1.0)
            weight = idf_dict.get(word, 1.0)

            # умножаем вектор на важность
            vectors.append(token.vector * weight)
            weights.append(weight)

    if len(vectors) > 0:
        # делим на сумму весов (нормализация)
        return np.sum(vectors, axis=0) / np.sum(weights)

    return np.zeros(nlp.vocab.vectors_length)

# ПРОВЕРКА (чтобы убедиться, что всё работает)
sample = "The movie was not very good but had amazing visuals."

print("Mean vector shape:", mean_pooling(sample).shape)
print("Max vector shape:", max_pooling(sample).shape)
print("Mean+Max shape:", mean_max_pooling(sample).shape)
print("TF-IDF shape:", tfidf_weighted(sample).shape)

Mean vector shape: (300,)
Max vector shape: (300,)
Mean+Max shape: (600,)
TF-IDF shape: (300,)


## C4. Train a simple classifier

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import numpy as np

# Подготовка данных
# перемешиваем данные, чтобы были и 0 и 1
shuffled_train_data = train_data.shuffle(seed=42)

# берём часть датасета (ускоряем обучение)
subset = shuffled_train_data.select(range(5000))

# тексты и метки
texts = [x["text"] for x in subset]
labels = [x["label"] for x in subset]  # 0 = negative, 1 = positive

# Преобразование текста в вектора
# используем mean pooling (из C3)
X = np.array([mean_pooling(t) for t in texts])
y = np.array(labels)


# Делим на train / validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,      # 20% данных → validation
    random_state=42     # фиксируем разбиение
)


# Обучение модели
# logistic regression — простой baseline
model = LogisticRegression(max_iter=1000)

# обучаем модель на train данных
model.fit(X_train, y_train)

# Оценка модели
# accuracy на обучении
train_acc = model.score(X_train, y_train)

# accuracy на валидации
val_acc = model.score(X_val, y_val)

print("Train accuracy:", train_acc)
print("Validation accuracy:", val_acc)

Train accuracy: 0.77425
Validation accuracy: 0.783


## C5. Evaluation

In [16]:
from sklearn.metrics import accuracy_score, confusion_matrix

# берём тест (перемешиваем для честности)
test_subset = test_data.shuffle(seed=42).select(range(2000))

# переводим текст → вектора
X_test = np.array([mean_pooling(x["text"]) for x in test_subset])
y_test = np.array([x["label"] for x in test_subset])

# предсказания модели
preds = model.predict(X_test)

# accuracy
test_acc = accuracy_score(y_test, preds)

print("\nTest accuracy:", test_acc)


# Confusion Matrix
cm = confusion_matrix(y_test, preds)

print("\nConfusion matrix:")
print(cm)


# Correct / Incorrect examples
correct = []
wrong = []

for i in range(len(test_subset)):
    text = test_subset[i]["text"]
    true = y_test[i]
    pred = preds[i]

    # правильные предсказания
    if true == pred and len(correct) < 5:
        correct.append((text[:150], true, pred))

    # ошибки
    if true != pred and len(wrong) < 5:
        wrong.append((text[:150], true, pred))


print("\nCorrect examples:")
for c in correct:
    print(c)

print("\nIncorrect examples:")
for w in wrong:
    print(w)


Test accuracy: 0.763

Confusion matrix:
[[785 215]
 [259 741]]

Correct examples:
('<br /><br />When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining King Lear story and of course Michelle Pfeiffer was i', np.int64(1), np.int64(1))
("This movie was so frustrating. Everything seemed energetic and I was totally prepared to have a good time. I at least thought I'd be able to stand it.", np.int64(0), np.int64(0))
('I was truly and wonderfully surprised at "O\' Brother, Where Art Thou?" The video store was out of all the movies I was planning on renting, so then I ', np.int64(1), np.int64(1))
('This movie spends most of its time preaching that it is the script that makes the movie, but apparently there was no script when they shot this waste ', np.int64(0), np.int64(0))
("This is a really sad, and touching movie! It deals with the subject of child abuse. It's really sad, but mostly a true story, because it happens every", np.int64(1), np.int64(1))

Incorrect 

Модель работает нормально, но ошибается в сложных случаях. Основные ошибки связаны с отрицанием и контекстом, потому что модель не учитывает порядок слов.

# Part D: (Code + Short Writing) Stress Test the Model: Where Order and Context Matter
В этой части я проверяю модель на специальных примерах, чтобы увидеть её ошибки. Это показывает, что модель не понимает порядок слов, отрицание и контекст.


## D1. Run your baseline on the contrast set

In [17]:
contrast_sentences = [
    # NEGATION
    "The movie was good.",
    "The movie was not good.",
    "I liked the film.",
    "I did not like the film.",
    "This was interesting.",
    "This was not interesting.",

    # WORD ORDER
    "Dog bites man.",
    "Man bites dog.",
    "She loves him.",
    "He loves her.",
    "The cat chased the mouse.",
    "The mouse chased the cat.",

    # AMBIGUITY
    "He went to the bank.",
    "He sat by the river bank.",

    # TRANSLATION
    "Flights from Astana to Almaty.",
    "Flights from Almaty to Astana.",

    # OWN
    "The movie was boring but had good acting.",
    "The movie was good but had boring acting."
]

# функция для перевода 0/1 → текст
def label_to_text(label):
    return "Positive" if label == 1 else "Negative"


for s in contrast_sentences:
    vec = mean_pooling(s)              # превращаем предложение в вектор
    pred = model.predict([vec])[0]     # предсказание модели

    print("Sentence:", s)
    print("Prediction:", label_to_text(pred))
    print("-" * 50)

Sentence: The movie was good.
Prediction: Negative
--------------------------------------------------
Sentence: The movie was not good.
Prediction: Negative
--------------------------------------------------
Sentence: I liked the film.
Prediction: Positive
--------------------------------------------------
Sentence: I did not like the film.
Prediction: Negative
--------------------------------------------------
Sentence: This was interesting.
Prediction: Positive
--------------------------------------------------
Sentence: This was not interesting.
Prediction: Positive
--------------------------------------------------
Sentence: Dog bites man.
Prediction: Negative
--------------------------------------------------
Sentence: Man bites dog.
Prediction: Negative
--------------------------------------------------
Sentence: She loves him.
Prediction: Positive
--------------------------------------------------
Sentence: He loves her.
Prediction: Positive
-------------------------------------

| Sentence | Prediction |
|----------|------------|
| The movie was good | Negative ❌ |
| The movie was not good | Negative |
| I liked the film | Positive |
| I did not like the film | Negative |
| This was interesting | Positive |
| This was not interesting | Positive ❌ |
| Dog bites man | Negative |
| Man bites dog | Negative ❌ |
| She loves him | Positive |
| He loves her | Positive ❌ |
| The cat chased the mouse | Positive |
| The mouse chased the cat | Positive ❌ |
| He went to the bank | Positive |
| He sat by the river bank | Positive ❌ |
| Flights from Astana to Almaty | Negative |
| Flights from Almaty to Astana | Negative ❌ |
| The movie was boring but had good acting | Negative |
| The movie was good but had boring acting | Negative ❌ |

Модель делает несколько типичных ошибок:

❌ не понимает отрицание
("good" и "not good" → одинаково)

❌ не видит порядок слов
("dog bites man" = "man bites dog")

❌ не понимает контекст
("bank" как банк и как берег)

❌ не справляется с контрастом
("good but boring")

👉 всё потому что:
мы просто усредняем слова

Усреднение теряет порядок слов, потому что рассматривает предложение как набор слов, а не последовательность. Также одно слово имеет один вектор, поэтому модель не понимает разные значения слова в разных контекстах.

RNN решает это, потому что читает слова по порядку и запоминает предыдущие слова (контекст).

# Part E: (Code + Short Writing) Choose One Bridge Mini-Track
В этой части я работаю с последовательностями (переводом), чтобы увидеть ограничения усреднения. Это показывает, что нужны модели, которые учитывают порядок слов.

## Option 1. Translation Mini-Track

In [18]:
# создаём небольшой датасет переводов
# ЗАЧЕМ: показать, что перевод — это задача последовательностей (sequence-to-sequence)
pairs = [
    ("good morning", "qaiyrly tan"),
    ("thank you", "rahmet"),
    ("how are you", "qalaisyn"),
    ("where is the station", "stantsiya qaida"),
    ("book a ticket", "bilet bron'dau"),
    ("open the door", "esikti ash"),
    ("close the window", "terezheni jab"),
    ("i love you", "men seni jaqsy koremin"),
    ("what is your name", "senin atyn kim"),
    ("see you later", "keiin koriskenshe")
]

# 1. Показываем пары
# ЗАЧЕМ: убедиться, что у нас есть вход (англ) и выход (перевод)
print("PAIRS")
for src, tgt in pairs:
    print(src, "->", tgt)


# 2. Tokenization + <SOS>, <EOS>
tokenized_pairs = []

for src, tgt in pairs:
    # разбиваем предложение на слова
    # ЗАЧЕМ: модель работает с токенами, а не с целыми строками
    src_tokens = src.lower().split()
    tgt_tokens = tgt.lower().split()

    # добавляем специальные токены
    # <SOS> — начало (модель понимает, откуда начинать)
    # <EOS> — конец (модель понимает, где остановиться)
    src_tokens = ["<SOS>"] + src_tokens + ["<EOS>"]
    tgt_tokens = ["<SOS>"] + tgt_tokens + ["<EOS>"]

    tokenized_pairs.append((src_tokens, tgt_tokens))


# ЗАЧЕМ: посмотреть, как текст превратился в последовательность
print("\nTOKENIZED")
for i in range(3):
    print(tokenized_pairs[i])


# 3. Длины последовательностей
# считаем длину предложений
# ЗАЧЕМ: показать, что предложения разной длины
src_lengths = [len(src) for src, _ in tokenized_pairs]
tgt_lengths = [len(tgt) for _, tgt in tokenized_pairs]

print("\nSource lengths:", src_lengths)
print("Target lengths:", tgt_lengths)

# это важно, потому что:
# нейросети требуют одинаковую длину входа


# 4. Padding
def pad(seq, max_len):
    # добавляем <PAD>, чтобы все предложения стали одинаковой длины
    # ЗАЧЕМ: чтобы можно было обрабатывать батчами
    return seq + ["<PAD>"] * (max_len - len(seq))


max_len = max(src_lengths)

print("\nPADDED BATCH")

for i in range(3):
    src, tgt = tokenized_pairs[i]

    padded_src = pad(src, max_len)
    padded_tgt = pad(tgt, max_len)

    print("SRC:", padded_src)
    print("TGT:", padded_tgt)
    print("-" * 40)


# ГЛАВНАЯ ИДЕЯ
# ЗАЧЕМ ВСЁ ЭТО:
# перевод — это не один вектор, а последовательность слов

# если просто усреднить слова (mean pooling),
# мы потеряем порядок слов → перевод будет невозможен

# поэтому нужны модели (RNN, LSTM),
# которые читают слова по порядку и запоминают контекст

PAIRS
good morning -> qaiyrly tan
thank you -> rahmet
how are you -> qalaisyn
where is the station -> stantsiya qaida
book a ticket -> bilet bron'dau
open the door -> esikti ash
close the window -> terezheni jab
i love you -> men seni jaqsy koremin
what is your name -> senin atyn kim
see you later -> keiin koriskenshe

TOKENIZED
(['<SOS>', 'good', 'morning', '<EOS>'], ['<SOS>', 'qaiyrly', 'tan', '<EOS>'])
(['<SOS>', 'thank', 'you', '<EOS>'], ['<SOS>', 'rahmet', '<EOS>'])
(['<SOS>', 'how', 'are', 'you', '<EOS>'], ['<SOS>', 'qalaisyn', '<EOS>'])

Source lengths: [4, 4, 5, 6, 5, 5, 5, 5, 6, 5]
Target lengths: [4, 3, 3, 4, 4, 4, 4, 6, 5, 4]

PADDED BATCH
SRC: ['<SOS>', 'good', 'morning', '<EOS>', '<PAD>', '<PAD>']
TGT: ['<SOS>', 'qaiyrly', 'tan', '<EOS>', '<PAD>', '<PAD>']
----------------------------------------
SRC: ['<SOS>', 'thank', 'you', '<EOS>', '<PAD>', '<PAD>']
TGT: ['<SOS>', 'rahmet', '<EOS>', '<PAD>', '<PAD>', '<PAD>']
----------------------------------------
SRC: ['<SOS>', 'how

Я создала небольшой датасет для перевода (английский → казахский), состоящий из 10 пар предложений. Я выполнила токенизацию, разбив предложения на слова, и добавила специальные токены <SOS> и <EOS>, чтобы обозначить начало и конец последовательности.

Я заметила, что предложения имеют разную длину, поэтому применила padding с помощью токена <PAD>, чтобы выровнять их. Это необходимо, потому что нейронные сети обрабатывают данные батчами и требуют одинаковый размер входных данных.

Этот эксперимент показывает, что перевод — это задача последовательностей. Усреднение эмбеддингов не подходит, потому что теряется порядок слов и структура. Правильный перевод зависит от порядка слов и контекста, которые нельзя представить одним усреднённым вектором.

# Part F: Follow-Up Questions for Understanding

## General questions

| Question (EN) | Вопрос (RU) | Answer (EN) | Ответ (RU) |
|--------------|------------|-------------|------------|
| What is the difference between a token, a type, and a vocabulary? | В чем разница между token, type и vocabulary? | A token is a word occurrence, a type is a unique word, and vocabulary is all unique words in the dataset. Example: “movie movie good” → 3 tokens, 2 types. | Token — это каждое слово в тексте, type — уникальное слово, vocabulary — все уникальные слова. Пример: “movie movie good” → 3 токена, 2 уникальных. |
| Why can pretrained embeddings help when the dataset is not very large? | Почему предобученные эмбеддинги помогают при маленьком датасете? | They already capture relationships between words. Example: “good” and “excellent” are close even with small data. | Они уже знают связи между словами. Пример: “good” и “excellent” близки даже при маленьком датасете. |
| Why is average pooling over word vectors order-invariant? | Почему average pooling не учитывает порядок слов? | Because it ignores word order. Example: “dog bites man” = “man bites dog”. | Потому что порядок слов игнорируется. Пример: “dog bites man” = “man bites dog”. |
| Give one example where static embeddings are useful and one where they are not enough. | Приведите пример, где статические эмбеддинги полезны и где их недостаточно. | Useful: sentiment classification. Not enough: translation. Example: translation requires word order. | Полезны: анализ тональности. Недостаточно: перевод. Пример: в переводе важен порядок слов. |
| What is the difference between a word embedding and a sentence embedding? | В чем разница между word embedding и sentence embedding? | Word embedding represents one word, sentence embedding represents a whole sentence. Example: “good” vs “this movie is good”. | Word embedding — одно слово, sentence embedding — всё предложение. Пример: “good” и “this movie is good”. |
| Why can the word "bank" cause trouble for static embeddings? | Почему слово "bank" вызывает проблемы? | Because it has multiple meanings but one vector. Example: bank (money) vs river bank. | Потому что у слова несколько значений, но один вектор. Пример: банк и берег реки. |
| If two sentences contain the same words in a different order, what will happen to a plain average embedding? Why is that a problem? | Что произойдет, если одинаковые слова в разном порядке? Почему это проблема? | They will have the same embedding even if meaning is different. Example: “I love you” and “you love I”. | Вектор будет одинаковым, хотя смысл разный. Пример: “I love you” и “you love I”. |
| What would you want the hidden state of an RNN to remember while reading a sentence? | Что должна помнить скрытая память RNN? | It should remember previous words and context. Example: “not good”. | Она должна помнить предыдущие слова и контекст. Пример: “not good”. |

---

## Translation mini-track questions

| Question (EN) | Вопрос (RU) | Answer (EN) | Ответ (RU) |
|--------------|------------|-------------|------------|
| Why do sequence-to-sequence models need <SOS> and <EOS> tokens? | Зачем нужны <SOS> и <EOS>? | They mark the start and end of a sequence. Example: model starts at <SOS> and stops at <EOS>. | Они обозначают начало и конец. Пример: модель начинает с <SOS> и заканчивает на <EOS>. |
| Why do source and target sentences usually need padding in batches? | Почему нужен padding? | Because sentences have different lengths and must be equal in a batch. Example: “hi” vs “how are you”. | Потому что предложения разной длины. Пример: короткое и длинное предложение. |
| Why is translation a sequence-to-sequence problem rather than a simple classification problem? | Почему перевод — это sequence-to-sequence задача? | Because output is a sequence, not one label. Example: “good morning” → “qaiyrly tan”. | Потому что результат — последовательность слов. Пример: перевод состоит из нескольких слов. |